In [1]:
import pandas as pd

df_lt = pd.read_csv("../data/processed/race_driver_labels.csv")

df_lt.head()

,raceId,driverId,constructorId,grid,positionOrder,points,statusId,year,round,circuitId,...,avgPitDuration_ms,driverName_x,constructorName_x,driverName_y,constructorName_y,driverName,constructorName,finishPosition,avgLapTime_s,constructorPoints
0,18,1,1,1,1,10.0,1,2008,1,1,...,0.0,Lewis Hamilton,McLaren,Lewis Hamilton,McLaren,Lewis Hamilton,McLaren,1,98.114069,14.0
1,18,2,2,5,2,8.0,1,2008,1,1,...,0.0,Nick Heidfeld,BMW Sauber,Nick Heidfeld,BMW Sauber,Nick Heidfeld,BMW Sauber,2,98.208517,8.0
2,18,3,3,7,3,6.0,1,2008,1,1,...,0.0,Nico Rosberg,Williams,Nico Rosberg,Williams,Nico Rosberg,Williams,3,98.254810,9.0
3,18,4,4,11,4,5.0,1,2008,1,1,...,0.0,Fernando Alonso,Renault,Fernando Alonso,Renault,Fernando Alonso,Renault,4,98.410293,5.0
4,18,5,1,3,5,4.0,1,2008,1,1,...,0.0,Heikki Kovalainen,McLaren,Heikki Kovalainen,McLaren,Heikki Kovalainen,McLaren,5,98.424655,14.0


In [2]:
required_cols = [
    "raceId",
    "avgLapTime_s",
    "qualifyingPosition",
    "constructorPoints",
    "pitStopCount"
]

missing = [c for c in required_cols if c not in df_lt.columns]
missing

[]

In [3]:
df_lt["lapTime_delta"] = (
    df_lt["avgLapTime_s"]
    - df_lt.groupby("raceId")["avgLapTime_s"].transform("mean")
)

In [4]:
df_lt["lapTime_delta"].describe()

count    1.104100e+04
mean    -2.741520e-16
std      1.186997e+01
min     -1.066625e+02
25%     -1.754876e+00
50%     -5.231658e-01
75%      5.634428e-01
max      5.932515e+02
Name: lapTime_delta, dtype: float64

In [5]:
features2 = [
    "qualifyingPosition",
    "constructorPoints",
    "pitStopCount"
]

df_model = df_lt.dropna(subset=features2 + ["lapTime_delta"]).copy()

X_lt = df_model[features2]
y_lt = df_model["lapTime_delta"]

In [6]:
from sklearn.model_selection import train_test_split

X_train_lt, X_test_lt, y_train_lt, y_test_lt = train_test_split(
    X_lt, y_lt, test_size=0.2, random_state=42
)

In [7]:
from sklearn.linear_model import LinearRegression

lr_lt = LinearRegression()
lr_lt.fit(X_train_lt, y_train_lt)

LinearRegression()

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_lt = lr_lt.predict(X_test_lt)

mae_lt = mean_absolute_error(y_test_lt, y_pred_lt)
rmse_lt = np.sqrt(mean_squared_error(y_test_lt, y_pred_lt))
r2_lt = r2_score(y_test_lt, y_pred_lt)

print("Average Lap Time — Linear Regression")
print(f"MAE: {mae_lt:.3f} seconds")
print(f"RMSE: {rmse_lt:.3f} seconds")
print(f"R²: {r2_lt:.3f}")

Average Lap Time — Linear Regression
MAE: 2.841 seconds
RMSE: 8.959 seconds
R²: 0.019


In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_lt = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_lt.fit(X_train_lt, y_train_lt)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [10]:
rf_lt = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_lt.fit(X_train_lt, y_train_lt)

y_pred_rf_lt = rf_lt.predict(X_test_lt)

print("Average Lap Time — Random Forest")
print(f"MAE: {mean_absolute_error(y_test_lt, y_pred_rf_lt):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_lt, y_pred_rf_lt)):.2f}")
print(f"R²: {r2_score(y_test_lt, y_pred_rf_lt):.3f}")

Average Lap Time — Random Forest
MAE: 3.38
RMSE: 9.32
R²: -0.063


In [11]:
X_lt.columns

Index(['qualifyingPosition', 'constructorPoints', 'pitStopCount'], dtype='object')

In [12]:
X_train_lt.equals(X_test_lt)

False